# YAML - Python

All 6 Python examples from [docs/yaml.md](https://platob.github.io/yggdryl/yaml/), in page order.

Generated by `scripts/build_docs_notebooks.py` from the blocks that
`scripts/check_docs_examples.py` compiles and runs, so every code cell below is
an example that passed. An edit here lives until the next build overwrites it.

The cells are unexecuted and run on any Python 3 kernel with the package
installed:

```console
pip install yggdryl
```

In [ ]:
from yggdryl import yaml

value = yaml.loads("symbol: AAPL\nquantity: 2\n")
assert value == {"symbol": "AAPL", "quantity": 2}

encoded = yaml.dumps(value)
assert isinstance(encoded, bytes)
assert yaml.loads(encoded) == value

# One document means one.
try:
    yaml.loads("id: 1\n---\nid: 2\n")
except ValueError as error:
    assert "expected one YAML document" in str(error)
else:
    raise AssertionError("a second document must be reported")

## Documents

In [ ]:
from yggdryl import yaml

documents = list(yaml.loads_all("id: 1\n---\nid: 2\n---\nnull\n"))
assert documents == [{"id": 1}, {"id": 2}, None]

encoded = yaml.dumps_all(documents)
assert b"\n---\n" in encoded
assert list(yaml.loads_all(encoded)) == documents

In [ ]:
import io

from yggdryl import yaml

documents = yaml.load_all(io.BytesIO(b"id: 1\n---\nitems: [1, 2\n"))
assert next(documents) == {"id": 1}

# The second document is malformed, and the iterator is done after saying so.
try:
    next(documents)
except ValueError as error:
    assert "at byte 23 (document byte 17)" in str(error)
else:
    raise AssertionError("the malformed document must be reported")

assert list(documents) == []

## Tags are read, never written

In [ ]:
from yggdryl import yaml

encoded = yaml.dumps({"payload": b"\x00\xff"})
assert b"!yggdryl" not in encoded
assert b'"$yggdryl": "bytes"' in encoded
assert yaml.loads(encoded) == {"payload": b"\x00\xff"}

In [ ]:
from yggdryl import yaml

# A machine tag on input is semantic.
assert yaml.loads("!yggdryl/bytes AP8=\n") == b"\x00\xff"

# An application tag is an annotation, so the node under it is the value.
assert yaml.loads("!vendor:quantity {value: 4}\n") == {"value": 4}

# A comment is not read either.
assert yaml.loads("# vendor:attacker\n!vendor:quantity {value: 4}\n") == {
    "value": 4
}

In [ ]:
from yggdryl import yaml

collision = {"$yggdryl": "bytes", "value": "AP8="}

encoded = yaml.dumps(collision)
assert b'"$yggdryl": "mapping"' in encoded
assert yaml.loads(encoded) == collision